In [2]:
!pip install yfinance tensorflow scikit-learn -q

In [3]:
import os
import pickle
import numpy as np
import yfinance as yf
import tensorflow as tf

from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.layers import (
    Dense,
    LSTM,
    Dropout,
    MultiHeadAttention,
    LayerNormalization,
    GlobalAveragePooling1D
)

In [4]:
STOCKS = {
    "AAPL": "Apple",
    "MSFT": "Microsoft",
    "GOOGL": "Google",
    "AMZN": "Amazon",
    "NVDA": "NVIDIA",
    "TSLA": "Tesla",
    "META": "Meta",
    "BRK-B": "Berkshire",
    "JPM": "JPMorgan",
    "V": "Visa"
}

WINDOW = 60
EPOCHS = 20
BATCH_SIZE = 32

FEATURES = [
    "Close",
    "Volume",
    "SMA20",
    "EMA20",
    "RSI",
    "Upper",
    "Lower"
]

os.makedirs("models", exist_ok=True)
os.makedirs("scalers", exist_ok=True)

In [5]:
def add_indicators(data):

    data["SMA20"] = data["Close"].rolling(20).mean()
    data["EMA20"] = data["Close"].ewm(span=20).mean()

    delta = data["Close"].diff()

    gain = delta.clip(lower=0).rolling(14).mean()
    loss = -delta.clip(upper=0).rolling(14).mean()

    rs = gain / loss

    data["RSI"] = 100 - (100 / (1 + rs))

    data["MA20"] = data["Close"].rolling(20).mean()
    data["STD"] = data["Close"].rolling(20).std()

    data["Upper"] = data["MA20"] + (2 * data["STD"])
    data["Lower"] = data["MA20"] - (2 * data["STD"])

    data.dropna(inplace=True)

    return data

In [6]:
def build_model(input_shape):

    inputs = tf.keras.Input(shape=input_shape)

    attention = MultiHeadAttention(
        num_heads=4,
        key_dim=32
    )(inputs, inputs)

    x = LayerNormalization()(inputs + attention)

    x = LSTM(
        64,
        return_sequences=True
    )(x)

    x = Dropout(0.2)(x)

    x = GlobalAveragePooling1D()(x)

    outputs = Dense(1)(x)

    model = tf.keras.Model(inputs, outputs)

    model.compile(
        optimizer="adam",
        loss="mse"
    )

    return model

In [7]:
for ticker, name in STOCKS.items():

    print("\n" + "="*50)
    print(f"Training {name} ({ticker})")
    print("="*50)

    raw = yf.download(
        ticker,
        period="2y",
        interval="1d",
        progress=False
    )

    if raw.empty:
        print("No data found")
        continue

    raw.columns = [
        col[0] if isinstance(col, tuple) else col
        for col in raw.columns
    ]

    data = add_indicators(raw)

    features = data[FEATURES]

    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(features)

    X = []
    y = []

    for i in range(WINDOW, len(scaled)):
        X.append(scaled[i-WINDOW:i])
        y.append(scaled[i, 0])

    X = np.array(X)
    y = np.array(y)

    print(f"Training Samples: {len(X)}")

    model = build_model((X.shape[1], X.shape[2]))

    model.fit(
        X,
        y,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=1
    )

    model.save_weights(
        f"models/{ticker}_model.weights.h5"
    )

    with open(
        f"scalers/{ticker}_scaler.pkl",
        "wb"
    ) as f:
        pickle.dump(scaler, f)

    print(f"Saved {ticker}")


Training Apple (AAPL)


/tmp/ipykernel_4918/1680337145.py:7: FutureWarning: YF.download() has changed argument auto_adjust default to True
  raw = yf.download(


Training Samples: 423
Epoch 1/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 5s 67ms/step - loss: 0.0627
Epoch 2/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 67ms/step - loss: 0.0196
Epoch 3/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 67ms/step - loss: 0.0130
Epoch 4/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 67ms/step - loss: 0.0101
Epoch 5/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 67ms/step - loss: 0.0085
Epoch 6/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 66ms/step - loss: 0.0074
Epoch 7/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 80ms/step - loss: 0.0061
Epoch 8/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 2s 107ms/step - loss: 0.0077
Epoch 9/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 100ms/step - loss: 0.0100
Epoch 10/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 66ms/step - loss: 0.0073
Epoch 11/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 65ms/step - loss: 0.0057
Epoch 12/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 66ms/step - loss: 0.0059
Epoch 13/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 67ms/step - loss: 0.0052
Epoch 14/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 67ms/step - loss: 0.0047
Epoch 15/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 67ms/

/tmp/ipykernel_4918/1680337145.py:7: FutureWarning: YF.download() has changed argument auto_adjust default to True
  raw = yf.download(


Training Samples: 423
Epoch 1/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 4s 64ms/step - loss: 0.1094
Epoch 2/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 67ms/step - loss: 0.0445
Epoch 3/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 66ms/step - loss: 0.0309
Epoch 4/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 65ms/step - loss: 0.0196
Epoch 5/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 81ms/step - loss: 0.0106
Epoch 6/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 2s 106ms/step - loss: 0.0076
Epoch 7/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 100ms/step - loss: 0.0072
Epoch 8/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 66ms/step - loss: 0.0097
Epoch 9/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 67ms/step - loss: 0.0078
Epoch 10/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 66ms/step - loss: 0.0063
Epoch 11/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 65ms/step - loss: 0.0074
Epoch 12/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 67ms/step - loss: 0.0071
Epoch 13/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 65ms/step - loss: 0.0061
Epoch 14/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 65ms/step - loss: 0.0057
Epoch 15/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 80ms/

...

Training Samples: 423
Epoch 1/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 5s 75ms/step - loss: 0.1925
Epoch 2/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 65ms/step - loss: 0.0201
Epoch 3/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 65ms/step - loss: 0.0172
Epoch 4/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 65ms/step - loss: 0.0155
Epoch 5/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 65ms/step - loss: 0.0149
Epoch 6/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 65ms/step - loss: 0.0130
Epoch 7/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 66ms/step - loss: 0.0117
Epoch 8/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 65ms/step - loss: 0.0104
Epoch 9/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 65ms/step - loss: 0.0096
Epoch 10/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 64ms/step - loss: 0.0117
Epoch 11/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 87ms/step - loss: 0.0098
Epoch 12/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 100ms/step - loss: 0.0089
Epoch 13/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 95ms/step - loss: 0.0074
Epoch 14/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 2s 65ms/step - loss: 0.0078
Epoch 15/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 64ms/s

/tmp/ipykernel_4918/1680337145.py:7: FutureWarning: YF.download() has changed argument auto_adjust default to True
  raw = yf.download(


Training Samples: 423
Epoch 1/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 5s 66ms/step - loss: 0.1079
Epoch 2/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 64ms/step - loss: 0.0128
Epoch 3/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 64ms/step - loss: 0.0114
Epoch 4/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 65ms/step - loss: 0.0091
Epoch 5/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 65ms/step - loss: 0.0092
Epoch 6/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 64ms/step - loss: 0.0079
Epoch 7/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 64ms/step - loss: 0.0089
Epoch 8/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 65ms/step - loss: 0.0070
Epoch 9/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 66ms/step - loss: 0.0066
Epoch 10/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 104ms/step - loss: 0.0063
Epoch 11/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 2s 108ms/step - loss: 0.0071
Epoch 12/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 2s 66ms/step - loss: 0.0079
Epoch 13/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 63ms/step - loss: 0.0086
Epoch 14/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 63ms/step - loss: 0.0061
Epoch 15/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 65ms/

/tmp/ipykernel_4918/1680337145.py:7: FutureWarning: YF.download() has changed argument auto_adjust default to True
  raw = yf.download(


Training Samples: 423
Epoch 1/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 5s 65ms/step - loss: 0.1171
Epoch 2/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 65ms/step - loss: 0.0198
Epoch 3/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 64ms/step - loss: 0.0151
Epoch 4/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 65ms/step - loss: 0.0130
Epoch 5/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 65ms/step - loss: 0.0116
Epoch 6/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 66ms/step - loss: 0.0109
Epoch 7/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 65ms/step - loss: 0.0106
Epoch 8/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 64ms/step - loss: 0.0106
Epoch 9/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 96ms/step - loss: 0.0095
Epoch 10/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 103ms/step - loss: 0.0094
Epoch 11/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 83ms/step - loss: 0.0095
Epoch 12/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 64ms/step - loss: 0.0090
Epoch 13/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 64ms/step - loss: 0.0088
Epoch 14/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 65ms/step - loss: 0.0086
Epoch 15/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 66ms/s

In [8]:
!zip -r stock_models.zip models scalers

  adding: models/ (stored 0%)
  adding: models/MSFT_model.weights.h5 (deflated 20%)
  adding: models/META_model.weights.h5 (deflated 20%)
  adding: models/JPM_model.weights.h5 (deflated 20%)
  adding: models/AMZN_model.weights.h5 (deflated 20%)
  adding: models/TSLA_model.weights.h5 (deflated 20%)
  adding: models/BRK-B_model.weights.h5 (deflated 20%)
  adding: models/AAPL_model.weights.h5 (deflated 20%)
  adding: models/NVDA_model.weights.h5 (deflated 20%)
  adding: models/GOOGL_model.weights.h5 (deflated 20%)
  adding: models/V_model.weights.h5 (deflated 19%)
  adding: scalers/ (stored 0%)
  adding: scalers/META_scaler.pkl (deflated 25%)
  adding: scalers/JPM_scaler.pkl (deflated 25%)
  adding: scalers/BRK-B_scaler.pkl (deflated 25%)
  adding: scalers/AMZN_scaler.pkl (deflated 24%)
  adding: scalers/NVDA_scaler.pkl (deflated 25%)
  adding: scalers/V_scaler.pkl (deflated 25%)
  adding: scalers/MSFT_scaler.pkl (deflated 25%)
  adding: scalers/GOOGL_scaler.pkl (deflated 25%)
  adding: s

In [9]:
from google.colab import files

files.download("stock_models.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
import os

print("Models:")
print(os.listdir("models"))

print("\nScalers:")
print(os.listdir("scalers"))

Models:
['MSFT_model.weights.h5', 'META_model.weights.h5', 'JPM_model.weights.h5', 'AMZN_model.weights.h5', 'TSLA_model.weights.h5', 'BRK-B_model.weights.h5', 'AAPL_model.weights.h5', 'NVDA_model.weights.h5', 'GOOGL_model.weights.h5', 'V_model.weights.h5']

Scalers:
['META_scaler.pkl', 'JPM_scaler.pkl', 'BRK-B_scaler.pkl', 'AMZN_scaler.pkl', 'NVDA_scaler.pkl', 'V_scaler.pkl', 'MSFT_scaler.pkl', 'GOOGL_scaler.pkl', 'TSLA_scaler.pkl', 'AAPL_scaler.pkl']
